# Classification: Naive Bayes (GaussianNB)

## Justification of Preprocessing Strategy

Unlike algorithms based on distances (such as KNN or SVM) or linear weights (such as Logistic Regression), Naive Bayes presents unique mathematical characteristics that influence data treatment:

### **Scale Invariance**
The GaussianNB calculates the probability of a class based on the mean and variance of each variable in isolation. Since the algorithm does not compare magnitudes between different variables, the original scale of the data is mathematically irrelevant to the final result.

### **Absence of Distance Metrics**
Since the model does not calculate Euclidean distances, there is no risk of variables with larger scales (e.g., Glucose) dominating variables with smaller scales (e.g., BMI).

### **Distribution Preservation**
Standardization (StandardScaler) is maintained only to ensure numerical stability and allow direct comparison with previous models. However, Normalization (MinMaxScaler) was deliberately excluded, as by "flattening" the data into a fixed interval [0, 1], it could distort the Gaussian curve in the presence of outliers, compromising classifier performance.

## Experiment Design

To validate these assumptions and optimize the model, we defined a tournament of 4 experiments:

### **Run 1: Absolute Baseline**
Training with original data (without any scaling) to demonstrate the algorithm's insensitivity to scale.

### **Run 2: Standardized Baseline**
Training with StandardScaler to serve as a comparison point for the following optimizations.

### **Run 3: Optuna Optimization**
Bayesian search focused on the var_smoothing hyperparameter, which controls variance smoothing to deal with data noise.

### **Run 4: GridSearchCV**
Exhaustive grid search over a logarithmic range on the same parameter, allowing comparison of efficiency between the two optimization methods.

In [7]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, accuracy_score, f1_score

# Configuration of MLflow

mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_NaiveBayes")

<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/NaivesBayes/mlruns/3'), creation_time=1777736386645, experiment_id='3', last_update_time=1777736386645, lifecycle_stage='active', name='Classification_NaiveBayes', tags={}, workspace='default'>

In [ ]:
df = pd.read_csv("C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/data/diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_nb_metrics(y_true, y_pred, duration):
    """Helper function to log metrics consistently across all runs"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: ABSOLUTE BASELINE (No Scalers)
# ---------------------------------------------------------
with mlflow.start_run(run_name="NB_NoScaler_Baseline"):
    model = GaussianNB()
    
    start_time = time.time()
    model.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred = model.predict(X_test)
    
    mlflow.log_params(model.get_params())
    mlflow.log_param("scaler", "None")
    mlflow.log_param("optimization", "none")
    
    log_nb_metrics(y_test, y_pred, duration)

# Prepare standardized data for remaining experiments
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# ---------------------------------------------------------
# RUN 2: STANDARDIZED BASELINE (With StandardScaler)
# ---------------------------------------------------------
with mlflow.start_run(run_name="NB_Standard_Baseline"):
    model = GaussianNB()
    
    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    duration = time.time() - start_time
    
    y_pred = model.predict(X_test_scaled)
    
    mlflow.log_params(model.get_params())
    mlflow.log_param("scaler", "Standardization")
    mlflow.log_param("optimization", "none")
    
    log_nb_metrics(y_test, y_pred, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA (Optimizing var_smoothing)
# ---------------------------------------------------------
def objective(trial):
    # var_smoothing controls the variance smoothing parameter
    var_s = trial.suggest_float("var_smoothing", 1e-11, 1e-1, log=True)
    m = GaussianNB(var_smoothing=var_s)
    m.fit(X_train_scaled, y_train)
    return recall_score(y_test, m.predict(X_test_scaled))

with mlflow.start_run(run_name="NB_Standard_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=20)
    duration = time.time() - start_time
    
    best_nb = GaussianNB(**study.best_params)
    best_nb.fit(X_train_scaled, y_train)
    y_pred = best_nb.predict(X_test_scaled)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("scaler", "Standardization")
    mlflow.log_param("optimization", "optuna")
    
    log_nb_metrics(y_test, y_pred, duration)

# ---------------------------------------------------------
# RUN 4: GRIDSEARCHCV (Optimizing var_smoothing)
# ---------------------------------------------------------
with mlflow.start_run(run_name="NB_Standard_GridSearch"):
    # Logarithmic grid search: test 50 values from 10^0 to 10^-11
    param_grid = {'var_smoothing': np.logspace(0, -11, num=50)}
    grid = GridSearchCV(GaussianNB(), param_grid, cv=5, scoring='recall', n_jobs=-1)
    
    start_time = time.time()
    grid.fit(X_train_scaled, y_train)
    duration = time.time() - start_time
    
    y_pred = grid.best_estimator_.predict(X_test_scaled)
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("scaler", "Standardization")
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_nb_metrics(y_test, y_pred, duration)


[I 2026-05-02 16:40:52,202] A new study created in memory with name: no-name-e43ff8c7-3758-45c1-84bd-47ca75bdc1b6
[I 2026-05-02 16:40:52,311] Trial 0 finished with value: 0.8225278872766921 and parameters: {'var_smoothing': 0.011592880995318966}. Best is trial 0 with value: 0.8225278872766921.
[I 2026-05-02 16:40:52,413] Trial 1 finished with value: 0.818166568816573 and parameters: {'var_smoothing': 0.08149778783009824}. Best is trial 0 with value: 0.8225278872766921.
[I 2026-05-02 16:40:52,517] Trial 2 finished with value: 0.8212698146439654 and parameters: {'var_smoothing': 0.000449106348566198}. Best is trial 0 with value: 0.8225278872766921.
[I 2026-05-02 16:40:52,622] Trial 3 finished with value: 0.8210182001174201 and parameters: {'var_smoothing': 5.675379085861799e-05}. Best is trial 0 with value: 0.8225278872766921.
[I 2026-05-02 16:40:52,725] Trial 4 finished with value: 0.8210182001174201 and parameters: {'var_smoothing': 9.301442343723328e-10}. Best is trial 0 with value: 0

## Naive Bayes Runs Comparison

| Run | Scaler | Optimization | var_smoothing | Recall | Accuracy | F1 | Fit Time (s) |
|---|---|---|---:|---:|---:|---:|---:|
| NB_NoScaler_Baseline | None | none | 1e-09 | 0.821018 | 0.81805 | 0.843261 | 0.081328 |
| NB_Standard_Baseline | Standardization | none | 1e-09 | 0.821018 | 0.81805 | 0.843261 | 0.080557 |
| NB_Standard_Optuna | Standardization | optuna | 0.011593 | 0.822528 | 0.82255 | 0.846782 | 2.171044 |
| NB_Standard_GridSearch | Standardization | GridSearchCV | 0.009541 | 0.823031 | 0.82255 | 0.846861 | 30.563020 |

### Analysis of Results

The four runs demonstrate the robustness of Gaussian Naive Bayes and its insensitivity to scaling:

- **Baseline Comparison**: Both baseline runs (with and without scaling) achieved identical results, confirming the theoretical expectation that GaussianNB is scale-invariant.

- **Optimization Impact**: Both optimization methods (Optuna and GridSearchCV) provided marginal improvements in recall and accuracy (~0.1-0.2%), with Optuna achieving this in significantly less computational time (2.17s vs 30.56s).

- **Selected Run for Deployment**: **NB_Standard_Optuna** offers the best balance between performance and efficiency:
  - Achieves optimal recall among optimized runs (0.822528)
  - Maintains high accuracy (0.82255)
  - Significantly faster training (2.17s) compared to GridSearchCV
  - Provides consistent and reliable predictions with minimal computational overhead